# vectorAnalysis — word2vec cosine similarity between personas and dictionary tokens

For a chosen **dictionary** (either the reward-model tokenizer vocabulary
`tokens_Ray2333…` or the BNC-style corpus `1_1_all_alpha.txt`) and the persona list
in `config/personas.yaml`, this notebook:

1. loads a **word2vec** model (Google-News 300d by default),
2. embeds every dictionary item and every persona name (multi-word phrases → mean of
   their in-vocabulary word vectors),
3. computes the **cosine similarity** between each persona and each dictionary item, and
4. writes **one CSV per persona** (plus a combined wide matrix) under
   `persona_analysis/vector_cosine_similarity/`.

### Which dictionary generated `persona_reward_model_scores`?
**`tokens_Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv`** — the reward model's tokenizer
vocabulary. The score files
(`data/persona_reward_model_scores/Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv`) and the
base-model logit files (`data/persona_base_model_logits/…`) share that dictionary's exact
schema (`token_id, token_name, token_decoded`) and its row count (131,636). The corpus
`1_1_all_alpha.txt` is an unrelated `Word/PoS/Freq` frequency list and was **not** used
for the RM sweep.

> **Alignment note:** if you want to line these cosine similarities up token-for-token
> against the RM scores / logits, keep `DICTIONARY = "tokens_Ray2333"` — that's the only
> dictionary those files are keyed on. `"corpora"` gives cleaner word2vec coverage but is
> not joinable to the RM/logit data.

## 0. Colab bootstrap

**Local Jupyter:** skip this cell — it detects that it isn't on Colab and does nothing.

**Colab:** run it first. It clones the repo, installs the extra deps this notebook needs
(`gensim`), and `chdir`s into the repo root so every relative path below resolves.

The repo is private, so the clone needs a GitHub token. Create a fine-grained PAT at
<https://github.com/settings/tokens?type=beta> scoped to
`puffables/rm-optpessimal-personas` with **Contents: Read-only**, then add it in the Colab
sidebar (key icon -> Secrets) as `GH_TOKEN` with "Notebook access" on — the same secret
`colab_run_pipeline.ipynb` uses.

**Memory / time:** `word2vec-google-news-300` is a ~1.6GB download and needs ~4GB RAM to
load, which a free-tier CPU runtime handles fine. The download does not survive a runtime
restart, so if you expect to restart, set `CACHE_VECTORS_ON_DRIVE = True` below to keep the
vectors on Drive instead. No GPU is needed for this notebook.

In [ ]:
# --- Colab bootstrap (no-op when running locally) ---
# Pure Python (no ! or %% magics) so the cell is also valid when the notebook is
# executed headlessly via nbconvert. Safe to re-run mid-session.
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_PATH = "puffables/rm-optpessimal-personas"
REPO_DIR  = "/content/rm-optpessimal-personas"
BRANCH    = "MFD-and-Analysis"     # branch to check out on Colab

# Cache the word2vec vectors on Google Drive so they survive a runtime restart.
# Leave False to re-download them each session (~2 min on a Colab connection).
CACHE_VECTORS_ON_DRIVE = False

def _run(*cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

if IN_COLAB:
    from google.colab import userdata

    # Private repo -> the clone is authenticated with the GH_TOKEN secret. The
    # token is only ever in the clone URL; the remote is rewritten immediately
    # after so it isn't left sitting in .git/config.
    if not os.path.exists(REPO_DIR):
        token = userdata.get("GH_TOKEN")
        _run("git", "clone", f"https://{token}@github.com/{REPO_PATH}.git", REPO_DIR)
        _run("git", "-C", REPO_DIR, "remote", "set-url", "origin",
             f"https://github.com/{REPO_PATH}.git")

    _run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
    _run("git", "-C", REPO_DIR, "checkout", BRANCH)
    os.chdir(REPO_DIR)

    # gensim 4.3.3 is the first release that imports cleanly against the scipy
    # Colab ships (scipy.linalg.triu was removed in scipy 1.13).
    _run(sys.executable, "-m", "pip", "install", "-q", "gensim>=4.3.3", "pyyaml")

    if CACHE_VECTORS_ON_DRIVE:
        # GENSIM_DATA_DIR *is* the cache directory (not its parent), and gensim
        # reads it at import time -- so it must be set here, before the
        # `import gensim.downloader` cell further down.
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["GENSIM_DATA_DIR"] = "/content/drive/MyDrive/gensim-data"
        os.makedirs(os.environ["GENSIM_DATA_DIR"], exist_ok=True)
        print("vector cache:", os.environ["GENSIM_DATA_DIR"])

print("running in Colab:", IN_COLAB)
print("working directory:", os.getcwd())

## 1. Config

Everything you'd normally want to change lives in this one block.

In [ ]:
from pathlib import Path

# ---- Which dictionary to embed ---------------------------------------------
#  "tokens_Ray2333" -> reward-model tokenizer vocabulary. This is what
#                      data/persona_reward_model_scores/ and
#                      data/persona_base_model_logits/ are keyed on, so choose
#                      this to align cosine-sim with RM scores / logits.
#  "corpora"        -> BNC-style word-frequency list (real English words; cleaner
#                      word2vec coverage, but NOT aligned to the RM/logit data).
DICTIONARY = "tokens_Ray2333"

# ---- word2vec model ---------------------------------------------------------
#  Any gensim-downloader model name (KeyedVectors interface) OR a local vectors
#  file. Google-News word2vec is 300d and ~1.6GB on first download (cached under
#  ~/gensim-data/). Lighter drop-in alternatives that use the identical interface:
#     "glove-wiki-gigaword-300"  (~380MB)
#     "glove-wiki-gigaword-50"   (~66MB, good for a quick smoke test)
W2V_MODEL        = "word2vec-google-news-300"
W2V_IS_LOCAL     = False   # True -> treat W2V_MODEL as a filesystem path
W2V_LOCAL_BINARY = True    # only used for a local file: True for Google-News .bin

# ---- personas ---------------------------------------------------------------
PERSONAS_YAML = "config/personas.yaml"

# ---- phrase -> vector aggregation -------------------------------------------
AGGREGATION       = "mean"   # mean of the in-vocab word vectors of a phrase
LOWERCASE_FALLBACK = True    # if exact-case miss, retry lower/capitalize/upper
# Function words dropped from multi-word phrases before averaging (so
# "a Black person" is embedded from {Black, person}, not diluted by "a").
DROP_WORDS = {"a", "an", "the", "of", "with", "without", "or", "and", "member"}

# ---- output -----------------------------------------------------------------
OUTPUT_DIR         = Path("persona_analysis/vector_cosine_similarity")
WRITE_PER_PERSONA  = True    # one cosine_sim__<persona>.csv per persona (as requested)
WRITE_WIDE_MATRIX  = True    # one combined tokens x personas wide CSV

# Resolve dictionary path + which column holds the surface text.
_DICT_SPECS = {
    "tokens_Ray2333": dict(
        path="data/dictionaries/tokens_Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv",
        read_kwargs=dict(),
        term_col="token_decoded",
        id_cols=["token_id", "token_name", "token_decoded"],
    ),
    "corpora": dict(
        path="data/corpora/1_1_all_alpha.txt",
        read_kwargs=dict(sep="\t", engine="python"),
        term_col="Word",
        id_cols=None,   # filled in after load (whatever columns the file has)
    ),
}
DICT_SPEC = _DICT_SPECS[DICTIONARY]
print(f"DICTIONARY = {DICTIONARY!r}  ->  {DICT_SPEC['path']}")
print(f"W2V_MODEL  = {W2V_MODEL!r}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")

## 2. Imports & phrase-embedding helpers

In [ ]:
import re
import numpy as np
import pandas as pd
import yaml

def slugify(text: str) -> str:
    """Match the slug convention used in the RM-score / logit column names."""
    text = text.strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")

def _lookup(word, kv, lowercase_fallback=True):
    cands = [word]
    if lowercase_fallback:
        cands += [word.lower(), word.capitalize(), word.upper()]
    for c in cands:
        if c in kv:                       # KeyedVectors __contains__
            return kv[c]
    return None

def embed_phrase(text, kv, drop_words=frozenset(), lowercase_fallback=True):
    """Embed a word or phrase as the mean of its in-vocabulary word vectors.

    Returns (vector | None, n_words_hit, n_words_total). Used for BOTH dictionary
    items (usually a single token) and multi-word persona names.
    """
    raw = str(text).strip()
    if not raw:
        return None, 0, 0
    words = [w for w in re.split(r"[^A-Za-z0-9']+", raw) if w]
    if not words:
        return None, 0, 0
    # Drop function words, but never drop *everything* (fall back to all words).
    content = [w for w in words if w.lower() not in drop_words] or words
    vecs = [v for w in content if (v := _lookup(w, kv, lowercase_fallback)) is not None]
    if not vecs:
        return None, 0, len(content)
    return np.mean(vecs, axis=0), len(vecs), len(content)

## 3. Load the word2vec model

First run downloads and caches the vectors under `~/gensim-data/`. Swap `W2V_MODEL` in the
config for a lighter GloVe model if you just want to sanity-check the pipeline.

In [ ]:
import gensim.downloader as api
from gensim.models import KeyedVectors

if W2V_IS_LOCAL:
    print(f"Loading local vectors from {W2V_MODEL} ...")
    kv = KeyedVectors.load_word2vec_format(W2V_MODEL, binary=W2V_LOCAL_BINARY)
else:
    print(f"Loading '{W2V_MODEL}' via gensim.downloader (cached in ~/gensim-data/) ...")
    kv = api.load(W2V_MODEL)

DIM = kv.vector_size
print(f"Loaded {len(kv):,} vectors, dim={DIM}")

## 4. Load the dictionary and embed every item

In [ ]:
dict_df = pd.read_csv(DICT_SPEC["path"], **DICT_SPEC["read_kwargs"])
dict_df.columns = [str(c).strip() for c in dict_df.columns]

term_col = DICT_SPEC["term_col"]
id_cols = DICT_SPEC["id_cols"] or list(dict_df.columns)
assert term_col in dict_df.columns, (
    f"Expected surface-text column {term_col!r}; found {list(dict_df.columns)}"
)

terms = dict_df[term_col].fillna("").astype(str).tolist()
print(f"{len(terms):,} dictionary items from column {term_col!r}")

# Embed each item, caching by surface string (many tokens repeat / are cheap).
emb = np.full((len(terms), DIM), np.nan, dtype=np.float32)
covered = np.zeros(len(terms), dtype=bool)
_cache = {}
for i, t in enumerate(terms):
    if t not in _cache:
        v, _, _ = embed_phrase(t, kv, DROP_WORDS, LOWERCASE_FALLBACK)
        _cache[t] = v
    v = _cache[t]
    if v is not None:
        emb[i] = v
        covered[i] = True

n_cov = int(covered.sum())
print(f"Embedded {n_cov:,} / {len(terms):,} items "
      f"({100*n_cov/len(terms):.1f}% coverage; rest are OOV -> NaN cosine).")

## 5. Load and embed the personas

In [ ]:
with open(PERSONAS_YAML) as f:
    personas = yaml.safe_load(f)["personas"]

persona_names = [p["name"] for p in personas]
persona_slugs = [slugify(n) for n in persona_names]

persona_vecs = {}
missed = []
for name, slug in zip(persona_names, persona_slugs):
    v, hit, total = embed_phrase(name, kv, DROP_WORDS, LOWERCASE_FALLBACK)
    if v is None:
        missed.append(name)
    else:
        persona_vecs[slug] = v

print(f"Embedded {len(persona_vecs)} / {len(persona_names)} personas.")
if missed:
    print("  Fully OOV (no in-vocab words):", missed)

## 6. Cosine similarity: each persona vs every dictionary item

Vectors are L2-normalized, so cosine similarity is a single matrix multiply. OOV
dictionary items get `NaN` (no vector to compare).

In [ ]:
# Normalized matrix of the covered dictionary vectors.
cov_idx = np.where(covered)[0]
D = emb[cov_idx]
D = D / np.linalg.norm(D, axis=1, keepdims=True)

kept_slugs = list(persona_vecs.keys())
P = np.vstack([persona_vecs[s] for s in kept_slugs]).astype(np.float32)
P = P / np.linalg.norm(P, axis=1, keepdims=True)

S = P @ D.T   # (n_personas, n_covered) cosine similarities
print("similarity matrix:", S.shape)

# Scatter back into full-length (token-aligned) columns, NaN for OOV items.
cosine_cols = {}
for r, slug in enumerate(kept_slugs):
    col = np.full(len(terms), np.nan, dtype=np.float32)
    col[cov_idx] = S[r]
    cosine_cols[slug] = col

## 7. Save — one CSV per persona (+ combined wide matrix)

Each per-persona CSV keeps the dictionary's id columns and appends `cosine_similarity`,
in original dictionary (token) order so it joins straight onto the RM-score / logit files.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base = dict_df[id_cols].copy()

written = []
if WRITE_PER_PERSONA:
    for slug in kept_slugs:
        out = base.copy()
        out["cosine_similarity"] = cosine_cols[slug]
        path = OUTPUT_DIR / f"cosine_sim__{slug}.csv"
        out.to_csv(path, index=False)
        written.append(path)
    print(f"Wrote {len(written)} per-persona CSVs to {OUTPUT_DIR}/")

if WRITE_WIDE_MATRIX:
    wide = base.copy()
    for slug in kept_slugs:
        wide[f"cosine__{slug}"] = cosine_cols[slug]
    wide_path = OUTPUT_DIR / f"cosine_sim__ALL_personas_wide__{DICTIONARY}.csv"
    wide.to_csv(wide_path, index=False)
    print(f"Wrote combined wide matrix -> {wide_path}")

# Small manifest recording what produced these files.
manifest = pd.DataFrame({
    "persona_name": persona_names,
    "persona_slug": persona_slugs,
    "embedded": [s in persona_vecs for s in persona_slugs],
})
manifest.to_csv(OUTPUT_DIR / "_manifest_personas.csv", index=False)
print("dictionary:", DICTIONARY, "| model:", W2V_MODEL,
      "| coverage: {:.1f}%".format(100*n_cov/len(terms)))

## 8. Sanity check — top items per persona

Do the nearest dictionary items look semantically sensible for each persona?

In [ ]:
def top_items(slug, k=12):
    col = cosine_cols[slug]
    order = np.argsort(np.where(np.isnan(col), -np.inf, col))[::-1][:k]
    return pd.DataFrame({
        "token": dict_df[term_col].values[order],
        "cosine": col[order],
    }).reset_index(drop=True)

for slug in kept_slugs[:3]:
    print(f"\n=== {slug} ===")
    print(top_items(slug).to_string(index=False))

## 9. Brainstorm — relating cosine similarity, logits, and reward-model scores

We now have **three token-level signals**, all keyed on the same
`token_id` and (for logits / RM scores) the same `<template>__<persona>` columns:

| Signal | Source | Meaning for a (persona, token) |
|---|---|---|
| **cosine similarity** | this notebook (word2vec) | how *semantically associated* the token is with the persona — a static, prompt-free prior |
| **base-model logit** | `data/persona_base_model_logits/…` | how likely the base LM is to *say* that token next, given the persona prompt |
| **reward-model score** | `data/persona_reward_model_scores/…` | how much the RM *rewards* that token as the completion, given the persona prompt |

Cosine similarity is the odd one out: it has **no template** and **no prompt** — one value
per (persona, token). Logits and RM scores exist per (template, persona, token). So most
comparisons below hold a *template* fixed (e.g. `greatest_self_id`) and broadcast the single
cosine column across it.

### A. Pairwise, within a single (persona, template)
Join the three signals on `token_id` and look at them two at a time:

- **Scatter matrix** (cosine↔logit, cosine↔RM, logit↔RM), one panel per pair, points = tokens.
- **Rank correlations** — Spearman ρ and Kendall τ (the RM/logit distributions are heavy-tailed;
  `scipy.stats` is already a dependency and used elsewhere in `persona_analysis`).
- **Top-k overlap / Jaccard** — of each persona's top-50 tokens by cosine vs by RM score vs by
  logit. "Does semantic association pick the same winners the RM does?"

### B. The interesting question: does semantics explain the RM *beyond* the base model?
The RM score and the base-model logit are naturally correlated (both are "Llama-3.2-3B sees
this prompt"). The value-add of cosine similarity is whether it explains the part of the RM
score the base model **doesn't**:

- **Residualize**: regress RM score on logit (per persona/template), then correlate the
  *residual* against cosine similarity. A positive relationship = "the RM over-rewards tokens
  semantically tied to the persona, over and above what the base model would say."
- **Partial correlation** corr(RM, cosine | logit) as the compact scalar version of the above.
- **Quadrant / 2D-binned heatmap**: logit on x, RM score on y, color = mean cosine per bin.
  Look at the **high-RM / low-logit** corner — tokens the RM loves that the base model wouldn't
  utter — and ask whether those are the high-cosine (persona-associated) tokens.

### C. The persona *shift* view (most aligned with the project's framing)
Everything above can be recomputed on **deltas from the matched baseline** (each template's
`baseline_column` in `persona_prompts.yaml`), which is exactly how `persona_analysis`
isolates the persona effect:

- Δlogit = persona logit − baseline logit; ΔRM = persona RM − baseline RM; and for cosine,
  a persona-specificity Δcos = cos(persona, token) − mean_over_personas cos(·, token).
- Then correlate **Δcos vs ΔRM** and **Δcos vs Δlogit**: "when a persona pulls a token up in
  semantic space, does the RM (and/or the base model) also pull it up?" This is the cleanest
  test of whether the RM's persona sensitivity tracks semantic identity association.

### D. Across-persona / across-token summaries
- **Correlation heatmap**: personas (rows) × {ρ(cos,RM), ρ(cos,logit), ρ(logit,RM)} (cols) — one
  glance at which personas the three signals agree on. Facet by persona `category`
  (race / gender / religion / political …).
- **MDS / 2D embedding of personas** from their cosine profiles (the wide matrix is already a
  persona × token feature matrix); overlay a marker sized by mean ΔRM to see if semantically
  clustered personas are also the ones the RM treats alike. (`sklearn.manifold.MDS` is already
  used in `analysis_support.py`.)
- **Token-level triptych** for a hand-picked token: bar-of-personas for cosine, logit, and RM
  side by side — e.g. does "church" score high on cosine, logit, and RM for the religious
  personas together?

### Recommended first cut
1. Pick one template (`greatest_self_id`) and one persona.
2. Join cosine + logit + RM on `token_id`; drop OOV (NaN cosine).
3. Scatter + Spearman for all three pairs **(A)**.
4. Residualize RM on logit, correlate residual vs cosine **(B)** — the headline number.
5. If promising, roll up to the per-persona correlation heatmap **(D)** and the Δ view **(C)**.

A follow-up notebook (`vectorAnalysis_relations.ipynb`) is the natural home for B–D once the
cosine CSVs from this notebook exist. Say the word and I'll scaffold it.